## 1. Импорты

In [ ]:
import os
import gc
import sys
import json
import toml
import datetime
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display
import matplotlib.pyplot as plt
import tqdm
from prefect.blocks.system import Secret
from prefect.variables import Variable
import plotly.io as pio
from scipy.stats import linregress

import lightgbm as lgb
from sklearn.metrics import roc_auc_score

pio.renderers.default = "plotly_mimetype"

## 2. Параметры

In [ ]:
secret_cdw = await Secret.load("pass-sb-analytics")
secret_space = await Secret.load("pass-space")
secret_sb_dirkaim = await Secret.load("pass-sb-dirkaim")

In [ ]:
SUBTASK = 'v1_oos'             # Подзадача (или версия, свободное наименование)
COMMENT = ""               # Текст-подсказка в свободной форме, пояснение для разработчика
SEED = 41                  # random_state

USERNAME_SPACE = "analytics"
USERNAME_CDW = "sb_analytics"
USERNAME_SB_DIRKAIM = "kovalenkovdy"
PASSWORD_SPACE = secret_space.get()
PASSWORD_CDW = secret_cdw.get()
PASSWORD_SB_DIRKAIM = secret_sb_dirkaim.get()


common_params = toml.load('common_params.toml')

AUTHOR = common_params['AUTHOR']        # Разработчик
PREFIX = common_params['PREFIX']        # Префикс
TASK_NUM = common_params['TASK_NUM']    # Номер задачи eservices
OUT_TABLE = common_params['OUT_TABLE']  # out_table
ML_TYPE = common_params['ML_TYPE']      # Тип ML-задачи: reg - регрессия, clf - классификация clf, mclf - мультирег,mclf - мультикласс
TARGET = 'target_25'        # Имя целевой переменной
TIME_COL = common_params['TIME_COL']    # Имя колонки с датами

PRIMARY_KEY = ['client_did', 'report_date', 'contract_uid']      # Объект предсказания, ключ

# Технические поля - не фичи, не id, не дата, для удаления
TECH_COLS = [
    'ub_clients_registration_dt_diff_report_date',
    'ub_clients_client_email',
    'ub_clients_mailing_consent',
    "card_report_date_diff_card_work_dt_aggmin", # ЛИК в фиче! 18.12.2025
    'transaction_total_trans_cnt_1m',
    'transaction_ratio_cnt_trans_1m_to_1m',
    'transaction_supermarkets_trans_cnt_12m'
]


# Пути (для чтения файлов)
PATH_TO_UTILS = common_params['PATH_TO_UTILS'] # Путь к кастомной библиотеке toolbox

# Пути (для сохранения файлов)
DATA_FOLDER = common_params['DATA_FOLDER']            # Папка с выборками
MODELS_FOLDER = common_params['MODELS_FOLDER']        # Файлы модели
FEATURES_FOLDER = common_params['FEATURES_FOLDER']    # Папка с фичами
REPORTS_FOLDER = f'{MODELS_FOLDER}/{SUBTASK}/report'  # Папка с отчетами

# Пути к файлам
PATH_TO_SAMPLE = f'./{DATA_FOLDER}/sample.parquet'    # Исходная выборка для разработки
PATH_MODELS_DATA = f"./{MODELS_FOLDER}/{SUBTASK}"     # Папка с файлми модели

In [ ]:
print(PATH_TO_SAMPLE, FEATURES_FOLDER, MODELS_FOLDER)
print(PATH_MODELS_DATA)
print(REPORTS_FOLDER)

In [ ]:
# Создаем необходимые директории
os.makedirs(MODELS_FOLDER, exist_ok=True)
os.makedirs(PATH_MODELS_DATA, exist_ok=True)
os.makedirs(REPORTS_FOLDER, exist_ok=True)

### Загрузка кастомного модуля

In [ ]:
sys.path.append(PATH_TO_UTILS)

In [ ]:
!pip install openpyxl # для from toolbox.modeling_utils.report

In [ ]:
from toolbox.modeling_utils import preprocessing 
from toolbox.modeling_utils import report
from toolbox.modeling_utils import feature_selection
from toolbox.modeling_utils import report_generator
from toolbox import oracle

## 3. Логгирование

In [ ]:
# Логгирование
import logging
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s %(levelname)s %(message)s",
    datefmt = "%Y-%m-%d %H:%M",
    handlers = [
        # logging.FileHandler("logfile.log"),
        logging.StreamHandler(),
    ])

In [ ]:
!pip install clearml

In [ ]:
os.environ["CLEARML_AUTO_LOG_ARTIFACTS"] = "0" 
os.environ["CLEARML_MONITOR_SYSTEM"]    = "0"
os.environ["CLEARML_AUTO_CONNECT_FRAMEWORKS"] = "0"

import clearml
from clearml import Task, Logger, Model

PROJECT_NAME = f"eservices-{TASK_NUM}"

clearml_task = Task.init(
    project_name=PROJECT_NAME,
    task_name=f"{PROJECT_NAME}_{SUBTASK}",
    auto_connect_streams=False,
    reuse_last_task_id=False
)

## 4 Чтение данных

In [ ]:
filenames = list(filter(
    lambda file: file.startswith(PREFIX.lower()) and file.endswith('.parquet'),
    os.listdir(FEATURES_FOLDER)
))
feature_paths = [f'{FEATURES_FOLDER}/{filename}' for filename in filenames]

sample = pd.read_parquet(PATH_TO_SAMPLE)

In [ ]:
# # Пути к фичам
# filenames = list(filter(lambda file: file.startswith(PREFIX.lower()) & file.endswith('.parquet'),  
#                         os.listdir(FEATURES_FOLDER)))
# feature_paths = [f"{FEATURES_FOLDER}/{filename}" for filename in filenames]

In [ ]:
# # Выборка для разработки
# sample = pd.read_parquet(PATH_TO_SAMPLE)
# display(sample.shape)
# display(sample.head(3))

# # Фичи
# df_features_lst = list()
# for path in tqdm.tqdm(feature_paths):
#     df_features_lst.append(pd.read_parquet(path))

In [ ]:
sample

In [ ]:
sample = sample.copy()
sample['report_date'] = pd.to_datetime(sample['report_date'])
sample = sample[sample['report_date'] >= '2025-09-01'].copy()

# OOS: случайные 20% строк сентября–декабря внутри каждого месяца и класса.
# Единица разбиения — клиент × контракт × дата; клиенты могут пересекаться.
# Январь остаётся целиком test для подбора, февраль — целиком OOT.
OOS_SHARE = 0.20
assert sample[PRIMARY_KEY].notna().all().all(), 'Пропуски в ключе выборки'
assert not sample.duplicated(PRIMARY_KEY).any(), 'Дубли полного ключа выборки'

train_period = sample['report_date'] < '2026-01-01'
split_rows = (
    sample.loc[train_period, PRIMARY_KEY + [TARGET]]
    .sort_values(PRIMARY_KEY)
    .reset_index(drop=True)
)
assert not split_rows.empty, 'Нет данных за сентябрь–декабрь'
assert split_rows[TARGET].isin([0, 1]).all(), 'Таргет должен быть заполнен и равен 0/1'

# Сохраняем полные ключи: индексы DataFrame могут измениться после merge.
# Размер OOS округляется до целого числа строк внутри каждой страты.
oos_rows = split_rows.groupby(
    [split_rows['report_date'].dt.to_period('M'), TARGET],
    observed=True,
    sort=True,
).sample(frac=OOS_SHARE, random_state=SEED)
oos_row_keys = pd.MultiIndex.from_frame(oos_rows[PRIMARY_KEY])
assert 0 < len(oos_row_keys) < len(split_rows), 'Недостаточно строк для разбиения'
del split_rows, oos_rows

def oos_mask(data):
    return pd.MultiIndex.from_frame(data[PRIMARY_KEY]).isin(oos_row_keys)

def training_mask(data):
    return ((data['report_date'] >= pd.Timestamp('2025-09-01'))
            & (data['report_date'] < pd.Timestamp('2026-01-01'))
            & ~oos_mask(data))


## 4. EDA

In [ ]:
# Распределение таргета в выборке
report_target_distrib = sample.groupby(TARGET).agg({TARGET: ['count', 'mean'], TIME_COL:['min', 'max']})
report_target_distrib

In [ ]:
#для отчета - Распределение таргета в выборке по месяцам
info_sample, fig_sample = report.counts_target_ratio_sample(sample, TIME_COL, TARGET)

Logger.current_logger().report_plotly(
    title="00401 Распределение таргета в выборке по месяцам",
    series="",
    figure=fig_sample)

In [ ]:
# Динамика таргета во времени
sample['year_month'] = sample[TIME_COL].dt.strftime('%Y-%m-01')
sample['year_month'] = pd.to_datetime(sample['year_month'])

In [ ]:
display(sample.pivot_table(values=['client_did'],
                index=['year_month'],
                columns=[TARGET],
                aggfunc='count',
                margins=False).T)

fig, ax = plt.subplots()
sample.pivot_table(values=['client_did'],
                index=['year_month'],
                columns=[TARGET],
                aggfunc='count',
                margins=False).plot(kind='barh', stacked=True, legend=True, rot=0,
                                    title='Количество клиентов по месяцам', ax=ax
                                   )
fig.tight_layout()
 
Logger.current_logger().report_matplotlib_figure(
    title='00402 Количество клиентов по месяцам',
    figure=fig,
    series=''
)

In [ ]:
display(sample.pivot_table(values=['contract_uid'],
                index=['year_month'],
                columns=[TARGET],
                aggfunc='count',
                margins=False).T)

fig, ax = plt.subplots()
sample.pivot_table(values=['contract_uid'],
                index=['year_month'],
                columns=[TARGET],
                aggfunc='count',
                margins=False).plot(kind='barh', stacked=True, legend=True, rot=0,
                                    title='Количество контрактов по месяцам', ax=ax
                                   )
fig.tight_layout()
 
Logger.current_logger().report_matplotlib_figure(
    title='00402 Количество контрактов по месяцам',
    figure=fig,
    series=''
)

## 5. Собираем ВДР и фичи вместе

In [ ]:
sample

In [ ]:
merged_df = sample.copy() 

In [ ]:
merged_df = merged_df.drop(['current_limit'], axis=1)

In [ ]:
%%time
import gc

MISSING_THRESHOLD = 0.8

for path, filename in tqdm.tqdm(zip(feature_paths, filenames), total=len(feature_paths)):
    feat_df = pd.read_parquet(path)

    merge_key = [c for c in PRIMARY_KEY if c in feat_df.columns]
    feat_df = feat_df.drop_duplicates(subset=merge_key, keep='first')

    # Фильтр пропусков применяется ниже на объединённых данных train.
    merged_df = merged_df.merge(feat_df, on=merge_key, how='left', copy=False)

    del feat_df
    gc.collect()

In [ ]:
# Чистим RAM
gc.collect()

In [ ]:
merged_df.info()
merged_df.shape

In [ ]:
merged_df.drop(columns=TECH_COLS, inplace=True, errors="ignore")

In [ ]:
gc.collect()

In [ ]:
display(merged_df.columns[:10])
all_features = merged_df.columns[4:] # Список всех доступных фичей

### Проверка

In [ ]:
display(merged_df.shape)
display(merged_df.head(3))

## 6. Отбор фичей (однофакторный анализ)


1. Убираем фичи с большой долей пропущенных значений
2. Убираем константные фичи
3. Убираем нестабильные фичи (`PSI > threshold`)
4. Разбиение на сегменты: TRAIN, TEST, OOT
5. Грубый отбор фичей (Оставляем ТОП-N фичей)

### 6.1 Убираем фичи с большой долей пропусков

In [ ]:
MISSING_THRESHOLD = 0.8  # Доля пропусков не более чем

missing_df = preprocessing.analyze_missing_values(merged_df.loc[training_mask(merged_df)], missing_threshold=MISSING_THRESHOLD)
missing_df.to_excel(f'{REPORTS_FOLDER}/missing_percentage.xlsx')

In [ ]:
merged_df = preprocessing.remove_columns_by_missing(merged_df, missing_df, 0.8)

In [ ]:
merged_df.shape

### 6.2 Убираем фичи с константными значениями (низкая дисперсия данных)

In [ ]:
res = preprocessing.find_constant_features(merged_df.loc[training_mask(merged_df)], threshold=0.95, verbose=False)
res['constant']

In [ ]:
merged_df.drop(res['constant'], axis = 1, inplace =True)

In [ ]:
merged_df.shape

In [ ]:
gc.collect()

In [ ]:
merged_df.to_parquet('checkpoint/merged_df_0925.parquet')

### 6.3 Отбор по PSI

In [ ]:
features = list(merged_df.drop(['report_date', 'client_did', 'contract_uid', 'target_50', 'target_75', 'target', TARGET, 'year_month'], axis=1).columns)

In [ ]:
gc.collect()

In [ ]:
%%time
# Отбор по PSI на всей доступной выборке, включая январь и февраль.
psi_result = preprocessing.calculate_feat_psi(merged_df[list(features) + ['year_month']], features, date_col='year_month')


In [ ]:
import pickle

with open('checkpoint/psi_result_0925.pkl', 'wb') as f:
    pickle.dump(psi_result, f)

In [ ]:
psi_result.head(5)

In [ ]:
# Фильтр 1: глобальное ограничение PSI на всей выборке
PSI_THRESHOLD = 0.15 # Убираем фичи, с PSI > PSI_THRESHOLD
# Фильтр 2: Ограничение тренда
# Если есть линейный тренд на рост PSI, то максимальный тренд PSI не более PSI_TREND
# PSI_TREND = 0.05
PSI_TREND = 0.10

In [ ]:
to_del_all = []
for col in psi_result.columns:
    max_psi = psi_result[col].max()
    lrgs = linregress(range(len(psi_result[col])), psi_result[col])
    if ((lrgs.slope > 0) & (lrgs.pvalue < 0.05) & (psi_result[col].max() > PSI_TREND)) or psi_result[col].max() >= PSI_THRESHOLD:
        to_del_all.append(col)

In [ ]:
psi_result[to_del_all].to_excel(f'{REPORTS_FOLDER}/psi_to_delete.xlsx')

In [ ]:
psi_result[ [   
    'current_limit',
    'contract_init_amt',
    'current_debt',
    # 'current_amt',
    'card_age_months',
    'current_interest_rate',
    'prol_prc_rt',
    'prc_rt_diff',
    'contract_amt_diff_init',
    'available_limit_by_debt',
    'util_current',
    'debt_daily_max_90d',
    'debt_daily_max_180d',
    'util_daily_max_90d',
    'util_daily_max_180d',
    'util_monthavg_avg_3m',
    'util_monthavg_avg_6m',
    'util_monthavg_avg_12m',
    'days_since_last_transaction',
	'risk_grade_last']]

In [ ]:
len(to_del_all), merged_df.shape

In [ ]:
to_del_all 

In [ ]:
'current_limit' in to_del_all

In [ ]:
merged_df.drop(to_del_all, axis=1, inplace=True)

### 6.4 Разбиение на сегменты: Train test split

Сегменты:
* `train` - Обучающая часть
* `test` - тестовая часть
* `oot` - в зависимости от типа валидации
* `other` - дубли, некорректные записи и тд

In [ ]:
merged_df = merged_df.drop(['contract_src_id'], axis = 1)

In [ ]:
merged_df

In [ ]:
df = merged_df[merged_df['report_date'] < '2026-02-01']
oot = merged_df[merged_df['report_date'] >= '2026-02-01'].copy()
train = df.loc[training_mask(df)].copy()
oos = df.loc[oos_mask(df)].copy()
test = df[df['report_date'] >= '2026-01-01'].copy()


In [ ]:
train.shape, oos.shape, test.shape, oot.shape


In [ ]:
del df
gc.collect()

In [ ]:
#для отчета - Размеры выборок и уровень таргета по месяцам
info_df, fig_df = report.counts_target_ratio(train, test, oot, TIME_COL, TARGET)
Logger.current_logger().report_plotly(
    title="00601 Размеры выборок и уровень таргета по месяцам",
    series="",
    figure=fig_df)

In [ ]:
info_df_to_log = info_df.copy()
info_df_to_log["month"] = pd.to_datetime(info_df_to_log["month"]).dt.strftime("%Y-%m-%d")
Logger.current_logger().report_table(
    title="00602 Размеры выборок и уровень таргета по месяцам",
    series="",
    table_plot=info_df_to_log
)
info_df

In [ ]:
# Проверка полноты разбиения и отсутствия общих строк train/OOS.
# Общие клиенты и контракты на разных датах допустимы.
assert len(train) + len(oos) + len(test) + len(oot) == len(sample)
assert not train.empty and not oos.empty
assert not train.duplicated(PRIMARY_KEY).any()
assert not oos.duplicated(PRIMARY_KEY).any()
assert len(pd.MultiIndex.from_frame(train[PRIMARY_KEY]).intersection(
    pd.MultiIndex.from_frame(oos[PRIMARY_KEY])
)) == 0
assert len(oos) == len(oos_row_keys)
assert ((oos['report_date'] >= '2025-09-01')
        & (oos['report_date'] < '2026-01-01')).all()
assert not oos_mask(test).any()
assert not oos_mask(oot).any()


## 7. Предварительный отбор фичей

In [ ]:
# Доработать
categorical_features = list(merged_df
                            .drop(columns=PRIMARY_KEY+['year_month','target','target_50','target_75', TARGET])
                            .select_dtypes(['category', 'object'])
                            .columns)
numerical_features = list(merged_df
                          .drop(columns=categorical_features + PRIMARY_KEY + ['year_month','target','target_50','target_75', TARGET])
                          .columns)

print(f'Categorical: {len(categorical_features)}')
print(f'Numerical: {len(numerical_features)}')

In [ ]:
features = categorical_features + numerical_features

In [ ]:
features

In [ ]:
%%time
PRESELECTION_TOP_N = 300  # Количество предварительно отобранных фичей

model, selected_features_100 = feature_selection.select_top_features(
    data=train[features + [TARGET]],
    targets=[TARGET],
    random_state=SEED,
    eval_set=test[features + [TARGET]],
    n_top=PRESELECTION_TOP_N,
    categorical_features=list(categorical_features)
)

In [ ]:
#для отчета - результаты обучения модели
res_df_top_N, roc_fig_top_N, pr_fig_top_N, lift_figs_top_N, cm_figs_top_N = report.model_results_plotly(train, test, oot, TARGET, features, model)

In [ ]:
preselected_features = selected_features_100[TARGET]['feature'].head(PRESELECTION_TOP_N).to_list()
preselected_features

## 8. Подбор оптимального количества фичей

In [ ]:
preselected_cat_features = list(train[preselected_features]
                            .select_dtypes(['category', 'object'])
                            .columns)

print(f'Categorical: {len(preselected_cat_features)}')
print(f'Numerical: {len(preselected_features) - len(preselected_cat_features)}')

In [ ]:
def recursive_feature_selection(
        train, test,
        preselected_features,
        preselected_cat_features,
        step=10,
        accuracy_threshold=0.01,
        min_features=20,
        random_state=SEED,
        target_col=TARGET,
        logger: Logger=None):
    """
    Подбирает оптимальное количество фичей через рекурсивное удаление;
    число фичей уменьшается с PRESELECTION_TOP_N до min_features;
    step: шаг кол-ва удаляемых на каждой итерации фичей: если int, то удаляется step фичей; если float, то удаляется доля step фичей; до достижения min_features;
    accuracy_threshold: максимально допустимая потеря ROC AUC относительно глобального максимума; выбирается минимальный набор в этом допуске.
    """
    if logger is None:
        logger = Logger.current_logger()

    candidates = []

    n_features = min(len(preselected_features), PRESELECTION_TOP_N)

    while n_features >= min_features:
        current_features = preselected_features[:n_features]
        X_tr = train[current_features]
        y_tr = train[target_col]
        X_te = test[current_features]
        y_te = test[target_col]

        model = lgb.LGBMClassifier(
            objective='binary', metric='auc',
            n_estimators=200,
            learning_rate=0.05,
            random_state=random_state,
            verbose=-1
        )
        model.fit(
            X_tr, y_tr,
            categorical_feature=[c for c in preselected_cat_features
                                 if c in current_features],
            eval_set=[(X_te, y_te)],
            eval_metric='auc',
            callbacks=[lgb.early_stopping(20, verbose=False)]
        )
        pred  = model.predict_proba(X_te)[:, 1]
        score = roc_auc_score(y_te, pred)

        logger.report_scalar(
            title='Динамика ROC AUC по количеству признаков',
            series='ROC AUC',
            iteration=n_features,
            value=score
        )

        logging.info(f"n_features={n_features:3d}  ROC AUC={score:.5f}")

        candidates.append({
            'n_features': n_features,
            'score': score,
            'features': current_features.copy(),
            'model': model,
        })

        if n_features == min_features:
            logging.info("Достигнут минимум признаков – завершаем подбор.")
            break

        if isinstance(step, int):
            n_features -= step
        else:
            n_features = int(n_features * (1 - step))

        if n_features < min_features:
            n_features = min_features

    # Допуск отсчитывается от глобального максимума, а не от предыдущего набора.
    max_score = max(item['score'] for item in candidates)
    eligible = [item for item in candidates
                if item['score'] >= max_score - accuracy_threshold]
    selected = min(eligible, key=lambda item: item['n_features'])
    best_features = selected['features']
    best_score = selected['score']
    best_model = selected['model']

    selection_results = pd.DataFrame([
        {'n_features': item['n_features'], 'roc_auc': item['score'],
         'loss_from_max': max_score - item['score'],
         'selected': item is selected}
        for item in candidates
    ])
    display(selection_results)
    logger.report_table(title='Выбор числа признаков', series='',
                        table_plot=selection_results)
    logging.info(f"Максимальный ROC AUC = {max_score:.5f}; "
                 f"выбран ROC AUC = {best_score:.5f} "
                 f"на {len(best_features)} признаках")
    return best_features, best_score, best_model


In [ ]:
%time
logger = clearml_task.get_logger()
best_features, best_score, best_model = recursive_feature_selection(
    train=train,
    test=test,
    preselected_features=preselected_features,
    preselected_cat_features=preselected_cat_features,
    step=20,
    accuracy_threshold=0.01,
    min_features=20,
    logger=logger)

In [ ]:
train = train[PRIMARY_KEY+['year_month', TARGET] + best_features]
test = test[PRIMARY_KEY+['year_month', TARGET] + best_features]
oot = oot[PRIMARY_KEY+['year_month', TARGET] + best_features]
oos = oos[PRIMARY_KEY+['year_month', TARGET] + best_features].copy()


## 9. Подбор гиперпараметров

In [ ]:
!pip install hyperopt

In [ ]:
from hyperopt import hp, fmin, tpe, Trials, STATUS_OK, space_eval


In [ ]:
# Число деревьев выбирается early stopping по AUC января (test).
MAX_ESTIMATORS = 3000
EARLY_STOPPING_ROUNDS = 100

def make_model_params(params, n_estimators=MAX_ESTIMATORS):
    return {
        'objective': 'binary',
        'metric': 'auc',
        'n_estimators': int(n_estimators),
        'learning_rate': float(params['learning_rate']),
        'max_depth': int(params['tree_shape']['max_depth']),
        'num_leaves': int(params['tree_shape']['num_leaves']),
        'min_child_samples': int(params['min_child_samples']),
        'reg_alpha': float(params['reg_alpha']),
        'reg_lambda': float(params['reg_lambda']),
        'subsample': float(params['subsample']),
        'subsample_freq': 1,
        'colsample_bytree': float(params['colsample_bytree']),
        'random_state': SEED,
        'n_jobs': -1,
        'verbosity': -1,
    }

def objective(params):
    model_params = make_model_params(params)
    model = lgb.LGBMClassifier(**model_params)
    model.fit(
        train[best_features], train[TARGET],
        eval_set=[(test[best_features], test[TARGET])],
        eval_metric='auc',
        callbacks=[lgb.early_stopping(
            EARLY_STOPPING_ROUNDS, first_metric_only=True, verbose=False
        )],
    )
    best_iteration = int(model.best_iteration_)
    y_pred = model.predict_proba(
        test[best_features], num_iteration=best_iteration
    )[:, 1]
    roc_auc = float(roc_auc_score(test[TARGET], y_pred))

    logger.report_scalar(
        title='Динамика ROC AUC по гиперпараметрам',
        series='ROC AUC', iteration=len(trials.trials), value=roc_auc
    )
    # OOS и OOT не используются ни в fit/eval_set, ни в loss поиска.
    return {
        'loss': -roc_auc,
        'status': STATUS_OK,
        'best_iteration': best_iteration,
        'model_params': model_params,
    }


In [ ]:
%%time
# Допустимые пары: num_leaves <= 2**max_depth.
space = {
    'learning_rate': hp.loguniform('learning_rate', np.log(0.02), np.log(0.10)),
    'subsample': hp.choice('subsample', [0.6, 0.8, 1.0]),
    'colsample_bytree': hp.choice('colsample_bytree', [0.6, 0.8, 1.0]),
    'tree_shape': hp.choice('tree_shape', [
        {'max_depth': 3, 'num_leaves': 7},
        {'max_depth': 4, 'num_leaves': 7},
        {'max_depth': 4, 'num_leaves': 15},
        {'max_depth': 5, 'num_leaves': 7},
        {'max_depth': 5, 'num_leaves': 15},
        {'max_depth': 5, 'num_leaves': 31},
    ]),
    'min_child_samples': hp.choice('min_child_samples', [100, 300, 500, 700, 1500]),
    'reg_alpha': hp.choice('reg_alpha', [0.0, 1.0, 5.0]),
    'reg_lambda': hp.choice('reg_lambda', [5.0, 15.0, 30.0]),
}

trials = Trials()
best = fmin(
    fn=objective,
    space=space,
    algo=tpe.suggest,
    max_evals=100,
    trials=trials,
    rstate=np.random.default_rng(SEED),
)

# hp.choice возвращает в best индексы вариантов, поэтому декодируем их.
best_search_params = space_eval(space, best)
best_trial_result = trials.best_trial['result']
search_results = pd.DataFrame([
    {'trial': trial['tid'],
     **trial['result']['model_params'],
     'best_iteration': trial['result']['best_iteration'],
     'roc_auc_test': -trial['result']['loss']}
    for trial in trials.trials if trial['result']['status'] == STATUS_OK
]).sort_values('roc_auc_test', ascending=False).reset_index(drop=True)
display(search_results)
search_results.to_csv(f'{REPORTS_FOLDER}/hyperopt_results.csv', index=False)
logger.report_table(title='Результаты подбора гиперпараметров',
                    series='', table_plot=search_results)


In [ ]:
# Финальная модель: параметры победителя и его лучшее число деревьев.
# Обучаем только на train; январь не добавляем к обучающим данным.
best_iteration = int(best_trial_result['best_iteration'])
best_params = make_model_params(best_search_params, n_estimators=best_iteration)
best_model = lgb.LGBMClassifier(**best_params)
best_model.fit(train[best_features], train[TARGET])

y_pred_best = best_model.predict_proba(test[best_features])[:, 1]
best_roc_auc = roc_auc_score(test[TARGET], y_pred_best)
# Проверяем, что финальное обучение воспроизводит выбранный trial.
np.testing.assert_allclose(
    best_roc_auc, -best_trial_result['loss'], rtol=0, atol=1e-6,
    err_msg='AUC финальной модели отличается от победителя Hyperopt'
)
print(f'Лучшие гиперпараметры: {best_params}')
print(f'Выбранное число деревьев: {best_iteration}')
print(f'ROC AUC на январской валидации: {best_roc_auc:.4f}')
if best_iteration >= MAX_ESTIMATORS:
    print('Лучший результат на верхней границе числа деревьев; проверьте лимит MAX_ESTIMATORS.')


In [ ]:
y_pred_oot = best_model.predict_proba(oot[best_features])[:, 1]
roc_auc_oot = roc_auc_score(oot[TARGET], y_pred_oot)
print(f"ROC AUC на oot выборке: {roc_auc_oot:.4f}")

In [ ]:
# Ручная модель params_reg удалена: best_model остаётся победителем Hyperopt.
# Все последующие метрики, сохранение модели и SHAP используют эту модель.


In [ ]:
# OOT уже оценён в ячейке выше, после финального обучения.
# Повторного обучения или замены best_model здесь нет.


In [ ]:
clearml_task.connect_configuration(configuration=best_model.get_params(),
                                   name='best_model.get_params()')

## 10. Сохранение артефактов

In [ ]:
# собираем проскоренную выборку - таблицу вида client_did | report_date | target | score | segment
y_pred_train = best_model.predict_proba(train[best_features])[:, 1]
roc_auc_train = roc_auc_score(train[TARGET], y_pred_train)

def create_df(df, y_pred, segment_name):
    result_df = df[PRIMARY_KEY+[TARGET]].copy()
    result_df['score'] = y_pred
    result_df['segment'] = segment_name

    return result_df

y_pred_oos = best_model.predict_proba(oos[best_features])[:, 1]
oos_df = create_df(oos, y_pred_oos, 'oos')

train_df = create_df(train, y_pred_train, 'train')
test_df = create_df(test, y_pred_best, 'test')
oot_df = create_df(oot, y_pred_oot, 'oot')

scored_df = pd.concat([train_df, oos_df, test_df, oot_df], ignore_index=True)

In [ ]:
# сохраняем train, test, oot выборки после отбора фичей и проскоренную выборку
train.to_parquet(f"{PATH_MODELS_DATA}/sample_train.parquet")
test.to_parquet(f"{PATH_MODELS_DATA}/sample_test.parquet")
oot.to_parquet(f"{PATH_MODELS_DATA}/sample_oot.parquet")
scored_df.to_parquet(f"{PATH_MODELS_DATA}/scored_df.parquet")

clearml_task.upload_artifact(name='train', artifact_object=train)
clearml_task.upload_artifact(name='test', artifact_object=test)
clearml_task.upload_artifact(name='oot', artifact_object=oot)
clearml_task.upload_artifact(name='scored_df', artifact_object=scored_df)
oos.to_parquet(f"{PATH_MODELS_DATA}/sample_oos.parquet")
clearml_task.upload_artifact(name='oos', artifact_object=oos)


In [ ]:
# Техническое имя модели
product = 'ptb_declip_25' # +- свободное наименование, главное отразить бизнес-процесс или бизнес-задачу, которую решает модель
monthYear = datetime.datetime.now().strftime('%m%y')

MODEL_NAME = f"{ML_TYPE}_{product}_{monthYear}_{SUBTASK}"

print(MODEL_NAME)

### pickle

In [ ]:
import pickle

In [ ]:
output_file = f"{PATH_MODELS_DATA}/{MODEL_NAME}.pkl"
with open(output_file, 'wb') as file: 
    pickle.dump(best_model, file) 

In [ ]:
clearml_task.upload_artifact(name=MODEL_NAME, artifact_object=best_model)

### pmml (для lighGBM)

In [ ]:
# пока может не работать
# from sklearn2pmml import sklearn2pmml, make_pmml_pipeline
# from sklearn2pmml.pipeline import PMMLPipeline
# from sklearn.pipeline import make_pipeline

In [ ]:
# output_file = f"{PATH_MODELS_DATA}/{MODEL_NAME}.pmml"
# pmml_model = make_pmml_pipeline(best_model, 
#                                 active_fields=best_model.feature_name_, 
#                                 target_fields=[TARGET])
# sklearn2pmml(pmml_model, output_file, with_repr=True)
# print(f"Модель сохранена в {output_file}")

## 11. Генерация отчета по модели

### Ключевые метрики

In [ ]:
#для отчета - Результаты обучения модели
res_df, roc_fig, pr_fig, lift_figs, cm_figs = report.model_results_plotly(train, test, oot, TARGET, best_features, best_model)

In [ ]:
lift_figs_titles = [
    "01103 Лифт train",
    "01104 Лифт test",
    "01105 Лифт oot"
]

for title, fig in zip(lift_figs_titles, lift_figs):
    Logger.current_logger().report_plotly(
        title=title,
        series=" ",
        figure=fig
    )

In [ ]:
Logger.current_logger().report_table(
    title="01106 Метрики",
    series="",
    table_plot=res_df
)
res_df

# OOS используется только для диагностики финальной модели, не для подбора.
def diagnostic_metrics(data):
    auc = (roc_auc_score(data[TARGET], data['score'])
           if data[TARGET].nunique() == 2 else np.nan)
    return {'N_samples': len(data), 'target_share': data[TARGET].mean(),
            'ROC': auc, 'gini': 2 * auc - 1}

diagnostic_rows = []
monthly_rows = []
for segment, frame in scored_df.groupby('segment', sort=False):
    diagnostic_rows.append({'segment': segment, **diagnostic_metrics(frame)})
    for month, group in frame.groupby(frame['report_date'].dt.to_period('M')):
        monthly_rows.append({'segment': segment, 'month': str(month),
                             **diagnostic_metrics(group)})
diagnostic_summary = pd.DataFrame(diagnostic_rows)
diagnostic_monthly = pd.DataFrame(monthly_rows)
display(diagnostic_summary)
display(diagnostic_monthly)
Logger.current_logger().report_table(
    title='01106 Диагностика Train/OOS/Test/OOT', series='',
    table_plot=diagnostic_summary)
Logger.current_logger().report_table(
    title='01106 Помесячная диагностика Train/OOS/Test/OOT', series='',
    table_plot=diagnostic_monthly)


In [ ]:
train['pred'] = best_model.predict_proba(train[best_features])[:,1]
test['pred'] = best_model.predict_proba(test[best_features])[:,1]
oot['pred'] = best_model.predict_proba(oot[best_features])[:,1]

train['decile'] = pd.qcut(-train['pred'], q = 10, labels=False, duplicates = 'drop') + 1
test['decile'] = pd.qcut(-test['pred'], q = 10, labels=False, duplicates = 'drop') + 1
oot['decile'] = pd.qcut(-oot['pred'], q = 10, labels=False, duplicates = 'drop') + 1

In [ ]:
assert (train['pred'] != 1).all()
assert (test['pred'] != 1).all()
assert (oot['pred'] != 1).all()

In [ ]:
#для отчета - Разбиение по децилям для всех выборок 
train_decile = report.decile_target(train, 'pred', TARGET, 'client_did')
test_decile = report.decile_target(test, 'pred', TARGET, 'client_did')
oot_decile = report.decile_target(oot, 'pred', TARGET, 'client_did')

In [ ]:
def plot_score_hist(df, score_col):
    plt.figure()
    plt.hist(df[score_col], bins=100)
    plt.show()

plot_score_hist(train, 'pred')
plot_score_hist(test, 'pred')
plot_score_hist(oot, 'pred')

In [ ]:
import pandas as pd
import numpy as np
import plotly.graph_objects as go


def gini_rate_plotly(
    data: pd.DataFrame,
    date_col: str,
    target_col: str,
    score_col: str,
    title: str = "Gini by months"):
    """
    Строит график с количеством записей (столбики) и Gini (линия) по месяцам.

    Параметры:
    ----------
    data : pd.DataFrame
        Исходные данные
    date_col : str
        Название колонки с датой
    target_col : str
        Название колонки с таргетом (0/1)
    score_col : str
        Название колонки с вероятностью/скором позитивного класса
    title : str, optional
        Заголовок графика (по умолчанию "Gini by months")

    Возвращает:
    -----------
    Tuple[pd.DataFrame, go.Figure]
        DataFrame с агрегированными данными и Plotly фигура

    Примечание:
    -----------
    Gini для каждого месяца = 2 * AUC - 1
    """
    df = data[[date_col, target_col, score_col]].copy()
    df[date_col] = pd.to_datetime(df[date_col], errors="coerce")
    df = df.dropna(subset=[date_col])

    # месяц = начало месяца
    df["month"] = df[date_col].dt.to_period("M").dt.to_timestamp()

    # cnt по месяцам
    agg_cnt = df.groupby("month", as_index=False).agg(cnt=(target_col, "size"))

    # gini по месяцам
    def _gini_month(gr: pd.DataFrame) -> float:
        y = gr[target_col].values
        s = gr[score_col].values
        if len(np.unique(y)) < 2:
            return np.nan  # AUC не определён, если в месяце один класс
        auc = roc_auc_score(y, s)
        return 2 * auc - 1

    gini = df.groupby("month").apply(_gini_month).reset_index(name="gini")

    df_agg = (
        agg_cnt.merge(gini, on="month", how="left")
        .sort_values("month")
        .reset_index(drop=True)
    )

    # --- Plotly ---
    fig = go.Figure()
    fig.add_bar(
        x=df_agg["month"],
        y=df_agg["cnt"],
        name="CNT",
        marker_color="royalblue",
        opacity=0.9,
    )
    fig.add_trace(
        go.Scatter(
            x=df_agg["month"],
            y=df_agg["gini"],
            mode="lines+markers",
            name="GINI",
            yaxis="y2",
            line=dict(width=3, color="red"),
            marker=dict(size=7, line=dict(width=1, color="white")),
        )
    )

    fig.update_layout(
        title=title,
        xaxis=dict(title=None, tickangle=45),
        yaxis=dict(
            title=dict(text="CNT", font=dict(color='royalblue')),
            tickformat=",.0f",
            color="royalblue",
        ),
        yaxis2=dict(
            title=dict(text="GINI", font=dict(color='red')),
            overlaying="y",
            side="right",
            range=[0, 1],
            tickformat=".2f",
            color="red",
        ),
        template="plotly_white",
        legend=dict(orientation="h", yanchor="top", y=-0.35, xanchor="center", x=0.5),
        margin=dict(t=70, r=70, b=70, l=70),
    )

    fig.show()

    return df_agg[["month", "cnt", "gini"]], fig

In [ ]:
#для отчета - GINI по месяцам
df_gini, fig_gini = report.gini_rate_plotly(pd.concat([train, test, oot], axis=0), TIME_COL, TARGET, 'pred')

Logger.current_logger().report_plotly(
    title="01107 Динамика gini по месяцам",
    series="",
    figure=fig_gini)

In [ ]:
best_features

### Важность фичей

In [ ]:
import_feat_df = pd.DataFrame({'feature_importance': best_model.feature_importances_, 
                               'feature_names': best_features,
                              }).sort_values(by=['feature_importance'], ascending=False)
Logger.current_logger().report_table(
    title="01108 Важность признаков",
    series="",
    table_plot=import_feat_df.reset_index(drop=True)
)
import_feat_df.reset_index()

### SHAP финальной модели

Диагностика `best_model` на случайной подвыборке OOS (до 5 000 контрактных строк).
OOS не используется для обучения или автоматического отбора признаков.
Вклады рассчитаны в шкале **log-odds** положительного класса: положительный SHAP
повышает прогноз относительно базового значения, отрицательный — понижает.

Если пакет `shap` не установлен, выполните `%pip install shap` в отдельной ячейке.
Расчёт: [LightGBM pred_contrib](https://lightgbm.readthedocs.io/en/stable/pythonapi/lightgbm.Booster.html#lightgbm.Booster.predict).
График: [SHAP beeswarm](https://shap.readthedocs.io/en/latest/generated/shap.plots.beeswarm.html).


In [ ]:
import shap

SHAP_SAMPLE_SIZE = 5000
SHAP_MAX_DISPLAY = 20

# Берём признаки строго в порядке, в котором обучена финальная модель.
shap_features = list(best_model.feature_name_)
assert not oos.empty, "Для SHAP нужна непустая OOS-выборка"
X_shap = oos[shap_features].sample(
    n=min(SHAP_SAMPLE_SIZE, len(oos)), random_state=SEED
).copy()

# Native TreeSHAP LightGBM: последний столбец — базовое значение.
# Не зависит от различий формата shap_values между версиями SHAP.
shap_contrib = np.asarray(best_model.booster_.predict(X_shap, pred_contrib=True))
assert shap_contrib.shape == (len(X_shap), len(shap_features) + 1)
shap_values = shap_contrib[:, :-1]
shap_base_values = shap_contrib[:, -1]
raw_scores = best_model.booster_.predict(X_shap, raw_score=True)
np.testing.assert_allclose(
    shap_base_values + shap_values.sum(axis=1), raw_scores,
    rtol=1e-5, atol=1e-6,
    err_msg="Сумма SHAP-вкладов не совпадает с raw score модели"
)

shap_explanation = shap.Explanation(
    values=shap_values,
    base_values=shap_base_values,
    data=X_shap,
    feature_names=shap_features,
)
shap_importance = pd.DataFrame({
    'feature': shap_features,
    'mean_abs_shap': np.abs(shap_values).mean(axis=0),
}).sort_values('mean_abs_shap', ascending=False).reset_index(drop=True)
shap_importance.to_csv(f'{REPORTS_FOLDER}/shap_importance_oos.csv', index=False)
display(shap_importance)
Logger.current_logger().report_table(
    title='01108 SHAP importance OOS', series='', table_plot=shap_importance
)


In [ ]:
# Средняя абсолютная величина вклада по признакам.
shap.plots.bar(shap_explanation, max_display=SHAP_MAX_DISPLAY, show=False)
shap_bar_fig = plt.gcf()
shap_bar_fig.suptitle('SHAP importance — OOS (log-odds)', y=1.02)
shap_bar_fig.savefig(f'{REPORTS_FOLDER}/shap_bar_oos.png',
                     dpi=150, bbox_inches='tight')
Logger.current_logger().report_matplotlib_figure(
    title='01108 SHAP bar OOS', series='', figure=shap_bar_fig
)
plt.show()
plt.close(shap_bar_fig)

# Распределение вкладов и направление влияния признаков.
shap.plots.beeswarm(shap_explanation, max_display=SHAP_MAX_DISPLAY, show=False)
shap_beeswarm_fig = plt.gcf()
shap_beeswarm_fig.suptitle('SHAP beeswarm — OOS (log-odds)', y=1.02)
shap_beeswarm_fig.savefig(f'{REPORTS_FOLDER}/shap_beeswarm_oos.png',
                          dpi=150, bbox_inches='tight')
Logger.current_logger().report_matplotlib_figure(
    title='01108 SHAP beeswarm OOS', series='', figure=shap_beeswarm_fig
)
plt.show()
plt.close(shap_beeswarm_fig)


### PSI

In [ ]:
psi_result[best_features].T.to_excel(f'{REPORTS_FOLDER}/psi_top40.xlsx')

In [ ]:
#для отчета - PSI фичей
_, psi_fig = report.psi_plotly(psi_result[best_features].T)
Logger.current_logger().report_plotly(
    title="01109 Динамика PSI признаков по месяцам",
    series="",
    figure=psi_fig)
psi_fig.show()

In [ ]:
psi_scores = preprocessing.calculate_feat_psi(pd.concat([train, test, oot], axis=0)[['pred', 'year_month']], ['pred'], date_col='year_month')
psi_scores

In [ ]:
#для отчета
_, psi_score_fig = report.psi_plotly(psi_scores.T, 'PSI score')
Logger.current_logger().report_plotly(
    title="01110 Динамика PSI скора по месяцам",
    series="",
    figure=psi_score_fig)
psi_score_fig.show()

In [ ]:
clearml_task.close()

### Маппинг фичей

In [ ]:
best_features

In [ ]:
train[best_features].info()

In [ ]:
secret_sb_dirkaim = (await Secret.load("pass-sb-dirkaim")).get()
USERNAME_SB_DIRKAIM = secret_sb_dirkaim["login"]
PASSWORD_SB_DIRKAIM = secret_sb_dirkaim["password"]

In [ ]:
%%time

#для  отчета
features_sql = ','.join(f"'{f}'" for f in best_features)

engine_sb_dirkaim = oracle.create_engine_cdw(USERNAME_SB_DIRKAIM, PASSWORD_SB_DIRKAIM)
with engine_sb_dirkaim.connect() as conn:
    description_df =  pd.read_sql(f"""select * from SB_DIRKAIM.FEATURE_REGISTRY where f_name in ({features_sql})""", conn)

In [ ]:
description_df = description_df[['f_name', 'f_desc', 'proc']]

#### Статистика по эффективности процедур

In [ ]:
from collections import defaultdict

config = await Variable.get("collect_feat_config")
stats = defaultdict(list)

# for key in config.keys():
#     for f in best_features:
#         if f.startswith(config[key]['suffix']):
#             stats[key].append(f)

# for key in stats.keys():
#     print(key, len(stats[key]))

for key, cfg in config.items():
    suffix = cfg["suffix"]
    for f in best_features:
        if f.startswith(suffix) or f.startswith(key):
            stats[key].append(f)

for key, feats in stats.items():
    print(f"{key}: {len(feats)}")

## HTML отчет

In [ ]:
# Поля, заполняемые вручную:
TITLE = "Модель вероятности, что ОД на 6 месяц будет больше 25% текущего лимита" # Название модели
DESCRIPTION = "Средняя ежедневная задолженность за 6‑й месяц (дни 151–180 после даты скоринга)." # Формализованное описание таргета 
TRAIN_TEST_PERIOD = "Сентябрь 2025 - январь 2026"
OOT_PERIOD = "Февраль 2026"
MODE = "offline model" # Режим работы (online/offline)
MODEL_TYPE = "LighGBM"
POPULATION_TEXT = "Клиенты с КК" # Описание выборки (напр. Оформили КК или получили СМС с предложением)

In [ ]:
# Создаём отчёт 
report_html = report_generator.Report_HTML(
    title=TITLE,
    description=DESCRIPTION,
    footer=f"Дата создания отчёта: {datetime.date.today().strftime('%d-%m-%Y')}" 
)

# === ВКЛАДКА 1: СПЕЦИФИКАЦИЯ МОДЕЛИ ===
spec_tab = report_html.add_tab("Спецификация модели")

# Строка 1x2: Основная информация
row1 = spec_tab.add_row("1x2")

# Основные характеристики
model_specs = {
    "Название": TITLE,
    "Тип задачи": ML_TYPE,
    "Режим работы": MODE,
    "Классификатор": MODEL_TYPE,
    "Версия модели": SUBTASK,
    "Разработчик модели": AUTHOR
}
row1.cells[0].add_kv(
    model_specs,
    title="Основные характеристики",
    caption="Ключевая информация о модели"
)

# Данные и выборки
data_specs = {
    "Источник данных": OUT_TABLE,
    "Популяция": POPULATION_TEXT,
    "Период обучения": TRAIN_TEST_PERIOD,
    "Период OOT": OOT_PERIOD,
}
row1.cells[1].add_kv(
    data_specs,
    title="Данные и выборки", 
    caption="Информация об источниках данных и временных периодах"
)

# Строка 1x2: Основная метрики
row2 = spec_tab.add_row(['1fr', '2fr'])
model_specs = best_model.get_params()

row2.cells[0].add_kv(
    model_specs,
    title="Параметры модели", 
    caption="Подобранные гиперпараметры"
)

rows = []
for name, df in [('Train', train), ('OOS', oos), ('Test', test), ('OOT', oot)]:
    total = len(df)
    positives = pd.to_numeric(df[TARGET], errors='coerce').sum()
    rate = positives / total
    rows.append([name, total, int(positives), f'{rate*100:.2f}%'])

cnt_df = pd.DataFrame(
    rows,
    columns = ['Выборка', 'Количество клиентов, шт.', 'Целевые события - таргет, шт.', 'Доля таргета']
)

row2.cells[1].add_dataframe(
    cnt_df,
    title="Кол-во клиентов и уровень таргета", 
    caption="Информация о выборке"
)

# === ВКЛАДКА 2: ВЫБОРКИ ===
sample_dfs_tab = report_html.add_tab("Данные для построения")

# Строка 1x2: График sample
row1 = sample_dfs_tab.add_row(['2fr', '1fr'])
row1.cells[0].add_plotly_figure(
    fig_sample,
    title="Динамика таргета во времени", 
    caption="Информация о выборке"
)
row1.cells[1].add_dataframe(
    info_sample,
    title="Динамика таргета во времени", 
    caption="Информация о выборке"
)

row2 = sample_dfs_tab.add_row("1x1")
row2.cells[0].add_plotly_figure(
    fig_df ,
    title="Динамика таргета во времени", 
    # caption="Информация о Train/Test/OOT"
)
row3 = sample_dfs_tab.add_row("1x1")
row3.cells[0].add_dataframe(
    info_df,
    title="Динамика таргета во времени", 
    caption="Информация о Train/Test/OOT"
)

# === ВКЛАДКА 3: МЕТРИКИ МОДЕЛИ ===
model_metrics_tab = report_html.add_tab("Метрики модели")

row1 = model_metrics_tab.add_row("1x1")
row1.cells[0].add_dataframe(
    res_df.T.reset_index(names='').drop(columns=['Recall', 'Precision', 'f05_score','f2_score']),
    title="Метрики качества", 
    caption="Показатели на Train/Test/OOT"
)
diagnostic_row = model_metrics_tab.add_row("1x1")
diagnostic_row.cells[0].add_dataframe(
    diagnostic_summary, title="Диагностика Train/OOS/Test/OOT")
monthly_row = model_metrics_tab.add_row("1x1")
monthly_row.cells[0].add_dataframe(
    diagnostic_monthly, title="Помесячная диагностика Train/OOS/Test/OOT")

row2 = model_metrics_tab.add_row(['2fr', '1fr'])
row2.cells[0].add_plotly_figure(
    fig_gini,
    title="Динамика GINI во времени", 
)
row2.cells[1].add_dataframe(
    df_gini,
    title="Динамика GINI во времени", 
)

row3 = model_metrics_tab.add_row("1x2")
row3.cells[0].add_plotly_figure(
        roc_fig,
        title="ROC-AUC")
row3.cells[1].add_plotly_figure(
        pr_fig,
        title="Precision-recall")

# row4 = model_metrics_tab.add_row("1x3")
# for i, fig in enumerate(cm_figs):
#     row4.cells[i].add_plotly_figure(
#         cm_figs[i],
#         title="Confusion matrix")

row4 = model_metrics_tab.add_row("1x3")
for i, fig in enumerate(lift_figs):
    row4.cells[i].add_plotly_figure(
        fig,
        title="Lift")

row5 = model_metrics_tab.add_row("1x3")
row5.cells[0].add_dataframe(
        train_decile.drop(columns=['target_0']),
        title="Таргет по децилям Train")
row5.cells[1].add_dataframe(
        test_decile.drop(columns=['target_0']),
        title="Таргет по децилям Test")
row5.cells[2].add_dataframe(
        oot_decile.drop(columns=['target_0']),
        title="Таргет по децилям OOT")

# === ВКЛАДКА 4: ФИЧИ И PSI ===
feat_tab = report_html.add_tab("Признаки и PSI")

row1 = feat_tab.add_row("1x1")
row1.cells[0].add_plotly_figure(
        psi_fig,
        title="PSI всех признаков модели по месяцам",)
row2 = feat_tab.add_row("1x1")
row2.cells[0].add_plotly_figure(
        psi_score_fig,
        title="PSI скоров модели по месяцам",)
row3 = feat_tab.add_row("1x1")
row3.cells[0].add_dataframe(
    pd.merge(import_feat_df, description_df, how='left', left_on='feature_names', right_on='f_name')[['feature_importance', 
                                                                                      'feature_names', 
                                                                                      'f_desc']],
    title="FeatureImp и расшифровка признаков", 
    caption="Источник описания - SB_DIRKAIM.FEATURE_REGISTRY"
)

report_html.save(f'{REPORTS_FOLDER}/{MODEL_NAME}_report.html')

## 12. Публикация модели в Model_Registry

In [ ]:
# Креды для подключения S3
s3_kwargs = (await Secret.load("s3-retail")).get()
# Пароль для работы с SB_DIRKAIM (!)
# cdw_pass = (await Secret.load("pass-sb-dirkaim")).get() 


from model_repository import ModelRepository
from model_repository.dao import OracleDAO
from model_repository.storage import MinioClient

engine = oracle.create_engine_cdw(
    USERNAME_SB_DIRKAIM, PASSWORD_SB_DIRKAIM
)
dao = OracleDAO(engine)
storage = MinioClient(**s3_kwargs)

# Инициализируем класс ModelRepository
repo = ModelRepository(storage=storage, dao=dao)

In [ ]:
from model_repository import RegisterRequest

# допустим, у тебя есть sklearn/catboost/lightgbm модель в памяти: best_model
req = RegisterRequest(
    model_name=MODEL_NAME,
    model_type="lightgbm",  # "catboost" | "lightgbm" | "sklearn" | "other"
    artifact_format="pickle",
    uri=f"retail/models/eservices-{TASK_NUM}/{SUBTASK}/{MODEL_NAME}.pkl",
    created_by=AUTHOR,
    description="Вероятность, что ОД на 6 месяц будет больше 25% текущего лимита", # Описание целевой переменной
    scoring_flg=0,  # Флаг что модель на активном скоринге
    monitoring_flg=0,  # Флаг что модель на мониторинге
    is_online=0,  # Флаг онлайн модели
    model_obj=best_model,  # авто-сериализуем, авто-извлечём важности
)

In [ ]:
# Регистрируем модель
meta = repo.register(req)
print(meta)

In [ ]:
# Проверка что модель зарегистрирована
assert repo.exists(MODEL_NAME)

# Сохранение данных

In [ ]:
s3_secret = (await Secret.load("s3-retail")).get()

In [ ]:
ACCESS_KEY = s3_secret["access_key"]
SECRET_KEY = s3_secret["secret_key"]

In [ ]:
from minio import Minio

client = Minio(
    endpoint="10.167.8.142:9000",
    access_key=ACCESS_KEY,
    secret_key=SECRET_KEY,
    secure=False
)    

In [ ]:
%%time
client.fput_object('retail', 
                   f'models/eservices-{TASK_NUM}/{SUBTASK}/train.paruqet',
                   f'{MODELS_FOLDER}/{SUBTASK}/sample_train.parquet')
client.fput_object('retail', 
                   f'models/eservices-{TASK_NUM}/{SUBTASK}/test.paruqet',
                   f'{MODELS_FOLDER}/{SUBTASK}/sample_test.parquet')
client.fput_object('retail', 
                   f'models/eservices-{TASK_NUM}/{SUBTASK}/oot.paruqet', 
                   f'{MODELS_FOLDER}/{SUBTASK}/sample_oot.parquet')
client.fput_object('retail', 
                   f'models/eservices-{TASK_NUM}/{SUBTASK}/scored_df.paruqet', 
                   f'{MODELS_FOLDER}/{SUBTASK}/scored_df.parquet')